# BRCA1 BRCT Variant Scoring Pipeline

Run this notebook in Google Colab to generate `variant_scores.csv` and `metrics.json`.

**Steps:**
1. Install dependencies
2. Create pipeline modules
3. Upload pre-fetched SGE data (`cache/sge_scores.csv`)
4. Run ESM-2 masked-marginal scoring (~2 min on GPU)
5. Join + compute AUROC
6. Download the output files

## 1. Install dependencies

In [ ]:
!pip install -q fair-esm torch pandas numpy scikit-learn

In [ ]:
import os
os.makedirs("pipeline", exist_ok=True)
os.makedirs("cache", exist_ok=True)
os.makedirs("data", exist_ok=True)

## 2. Create pipeline modules

In [ ]:
%%writefile pipeline/__init__.py
"""BRCA1 BRCT variant scoring pipeline."""

In [ ]:
%%writefile pipeline/config.py
"""Pipeline constants for BRCA1 BRCT variant scoring.

Canonical coordinate system: full-protein BRCA1 numbering (UniProt P38398).
BRCT tandem-repeat domain = residues 1646-1859 (inclusive, 214 residues).
"""

BRCT_START = 1646
BRCT_END = 1859
BRCT_LENGTH = BRCT_END - BRCT_START + 1

# Verified against https://rest.uniprot.org/uniprotkb/P38398.fasta
# Residues 1646-1863 (BRCT domain + 4 C-terminal residues to end of protein).
# We score only positions 1646-1859 (first 214 residues); the rest is context.
_BRCT_CONTEXT_SEQ = (
    "VNKRMSMVVSGLTPEEFMLVYKFA"
    "RKHHITLTNLITEETTHVVMKTDA"
    "EFVCERTLKYFLGIAGGKWVVSYF"
    "WVTQSIKERKMLNEHDFEVRGDVV"
    "NGRNHQGPKRARESQDRKIFRGLE"
    "ICCYGPFTNMPTDQLEWMVQLCGA"
    "SVVKELSSFTLGTGVHPIVVVQPD"
    "AWTEDNGFHAIGQMCEAPVVTREW"
    "VLDSVALYQCQELDTYLIPQIP"
    "HSHY"
)

BRCT_WT_SEQ = _BRCT_CONTEXT_SEQ[:BRCT_LENGTH]
assert len(BRCT_WT_SEQ) == BRCT_LENGTH, f"WT sequence length {len(BRCT_WT_SEQ)} != expected {BRCT_LENGTH}"
BRCT_CONTEXT_SEQ = _BRCT_CONTEXT_SEQ

AMINO_ACIDS = sorted("ACDEFGHIKLMNPQRSTVWY")

SGE_FUNC_THRESHOLD = -0.748
SGE_LOF_THRESHOLD = -1.328

def classify_sge(score: float) -> str:
    if score > SGE_FUNC_THRESHOLD:
        return "FUNC"
    elif score >= SGE_LOF_THRESHOLD:
        return "INT"
    else:
        return "LOF"

ESM_MODEL_PRIMARY = "esm2_t33_650M_UR50D"
ESM_MODEL_FALLBACK = "esm2_t30_150M_UR50D"

CACHE_DIR = "cache"
DATA_DIR = "data"
ESM_CACHE_FILE = "esm_scores.csv"
SGE_CACHE_FILE = "sge_scores.csv"
VARIANT_SCORES_FILE = "variant_scores.csv"
METRICS_FILE = "metrics.json"

def repo_root():
    import os
    return os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

def cache_path(filename):
    import os
    d = os.path.join(repo_root(), CACHE_DIR)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, filename)

def data_path(filename):
    import os
    d = os.path.join(repo_root(), DATA_DIR)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, filename)

In [ ]:
%%writefile pipeline/esm_scoring.py
"""ESM-2 masked-marginal scoring for BRCA1 BRCT domain variants."""

import os
import sys
import time

import numpy as np
import pandas as pd
import torch

from pipeline.config import (
    AMINO_ACIDS, BRCT_CONTEXT_SEQ, BRCT_END, BRCT_LENGTH,
    BRCT_START, BRCT_WT_SEQ, ESM_CACHE_FILE,
    ESM_MODEL_FALLBACK, ESM_MODEL_PRIMARY, cache_path,
)


def load_esm_model():
    import esm
    for model_name in [ESM_MODEL_PRIMARY, ESM_MODEL_FALLBACK]:
        try:
            print(f"Loading {model_name}...")
            load_fn = getattr(esm.pretrained, model_name, None)
            if load_fn is not None:
                model, alphabet = load_fn()
            else:
                model, alphabet = esm.pretrained.load_model_and_alphabet(model_name)
            model.eval()
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            model = model.to(device)
            n_params = sum(p.numel() for p in model.parameters()) / 1e6
            print(f"  Loaded on {device} ({n_params:.0f}M params)")
            return model, alphabet, model_name, device
        except Exception as e:
            print(f"  Failed to load {model_name}: {e}")
            continue
    raise RuntimeError("Could not load any ESM-2 model")


def masked_marginal_scores(model, alphabet, sequence, brct_start, brct_length, device):
    batch_converter = alphabet.get_batch_converter()
    _, _, wt_tokens = batch_converter([("brct", sequence)])
    wt_tokens = wt_tokens.to(device)

    results = []
    t0 = time.time()

    for i in range(brct_length):
        pos = brct_start + i
        wt_aa = sequence[i]
        assert wt_aa == BRCT_WT_SEQ[i], f"WT mismatch at {pos}"

        masked_tokens = wt_tokens.clone()
        token_idx = i + 1
        masked_tokens[0, token_idx] = alphabet.mask_idx

        with torch.no_grad():
            logits = model(masked_tokens)["logits"]

        log_probs = torch.log_softmax(logits[0, token_idx], dim=-1)
        wt_token_id = alphabet.get_idx(wt_aa)
        wt_log_prob = log_probs[wt_token_id].item()

        for mut_aa in AMINO_ACIDS:
            if mut_aa == wt_aa:
                continue
            mut_token_id = alphabet.get_idx(mut_aa)
            mut_log_prob = log_probs[mut_token_id].item()
            esm_score = mut_log_prob - wt_log_prob
            results.append({
                "position": pos, "wt_aa": wt_aa,
                "mut_aa": mut_aa, "esm_score": round(esm_score, 6),
            })

        elapsed = time.time() - t0
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        eta = (brct_length - i - 1) / rate if rate > 0 else 0
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Position {pos} ({i+1}/{brct_length}) | {elapsed:.0f}s elapsed | ETA {eta:.0f}s")

    print(f"  Done: {brct_length} positions, {len(results)} variants in {time.time()-t0:.1f}s")
    return results


def run_esm_scoring(force=False):
    out = cache_path(ESM_CACHE_FILE)
    if os.path.exists(out) and not force:
        print(f"Loading cached ESM scores from {out}")
        return pd.read_csv(out)

    model, alphabet, model_name, device = load_esm_model()
    print(f"Scoring {BRCT_LENGTH} positions ({BRCT_START}-{BRCT_END})...")
    results = masked_marginal_scores(model, alphabet, BRCT_CONTEXT_SEQ, BRCT_START, BRCT_LENGTH, device)

    df = pd.DataFrame(results)
    df.to_csv(out, index=False)
    print(f"Saved to {out}")
    median = df["esm_score"].median()
    print(f"  Median ESM score: {median:.3f} (should be negative)")
    return df

In [ ]:
%%writefile pipeline/join_and_evaluate.py
"""Join ESM-2 scores with SGE data and compute metrics."""

import json
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from pipeline.config import (
    ESM_CACHE_FILE, METRICS_FILE, SGE_CACHE_FILE,
    VARIANT_SCORES_FILE, cache_path, data_path, ESM_MODEL_PRIMARY,
)


def run_join_and_evaluate():
    print("=" * 60)
    print("Join & Evaluate")
    print("=" * 60)

    esm_df = pd.read_csv(cache_path(ESM_CACHE_FILE))
    print(f"ESM: {len(esm_df)} variants")

    sge_path = cache_path(SGE_CACHE_FILE)
    if os.path.exists(sge_path):
        sge_df = pd.read_csv(sge_path)
        print(f"SGE: {len(sge_df)} variants")
    else:
        sge_df = pd.DataFrame(columns=["position", "wt_aa", "mut_aa", "sge_score", "sge_class"])
        print("SGE: no data")

    join_cols = ["position", "wt_aa", "mut_aa"]
    if len(sge_df) == 0:
        merged = esm_df.copy()
        merged["sge_score"] = np.nan
        merged["sge_class"] = ""
    else:
        esm_df["position"] = esm_df["position"].astype(int)
        sge_df["position"] = sge_df["position"].astype(int)
        merged = esm_df.merge(sge_df[["position", "wt_aa", "mut_aa", "sge_score", "sge_class"]], on=join_cols, how="left")

    merged["sge_class"] = merged["sge_class"].fillna("")
    merged = merged[["position", "wt_aa", "mut_aa", "esm_score", "sge_score", "sge_class"]]
    n_with_sge = int(merged["sge_score"].notna().sum())
    print(f"Joined: {len(merged)} total, {n_with_sge} with SGE")

    auroc = float("nan")
    mask = merged["sge_class"].isin(["LOF", "FUNC"])
    subset = merged[mask].copy()
    if len(subset) >= 10:
        y_true = (subset["sge_class"] == "LOF").astype(int).values
        esm = subset["esm_score"].values
        mean_lof = esm[y_true == 1].mean()
        mean_func = esm[y_true == 0].mean()
        print(f"  Mean ESM -- LOF: {mean_lof:.3f}, FUNC: {mean_func:.3f}")
        auroc = roc_auc_score(y_true, -esm)
        if auroc < 0.5:
            auroc = roc_auc_score(y_true, esm)
        auroc = round(auroc, 4)
        print(f"  AUROC = {auroc}")
    else:
        print("  Too few LOF/FUNC for AUROC")

    csv_out = data_path(VARIANT_SCORES_FILE)
    merged.to_csv(csv_out, index=False)
    print(f"Wrote {csv_out}")

    metrics = {"auroc": auroc, "n_variants": len(merged), "n_with_sge": n_with_sge, "model": ESM_MODEL_PRIMARY}
    metrics_out = data_path(METRICS_FILE)
    with open(metrics_out, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Wrote {metrics_out}: {metrics}")
    return merged, metrics

## 3. Upload pre-fetched SGE data

The SGE experimental data was fetched locally from Findlay et al. 2018 Supplementary Table 3
(the MaveDB deposit only has nucleotide-level HGVS, not amino acid).

Upload `cache/sge_scores.csv` from your local repo when prompted.

In [ ]:
import pandas as pd

try:
    from google.colab import files
    print("Upload cache/sge_scores.csv from your local repo:")
    uploaded = files.upload()
    # Move to cache/
    for fname in uploaded:
        with open(f"cache/{fname}", "wb") as f:
            f.write(uploaded[fname])
        print(f"  Saved to cache/{fname}")
except ImportError:
    print("Not in Colab — assuming cache/sge_scores.csv already exists")

# Verify
sge = pd.read_csv("cache/sge_scores.csv")
print(f"\nSGE data: {len(sge)} variants")
print(f"Position range: {sge['position'].min()}-{sge['position'].max()}")
print(f"Class distribution:\n{sge['sge_class'].value_counts().to_string()}")

## 4. Check GPU & run ESM-2 scoring

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import time
t0 = time.time()

print("[Stage 1] ESM-2 masked-marginal scoring...")
from pipeline.esm_scoring import run_esm_scoring
esm_df = run_esm_scoring(force=False)
print(f"  {len(esm_df)} variant scores")
print(f"  Done in {time.time()-t0:.1f}s")

## 5. Join & evaluate

In [ ]:
print("[Stage 2] Join & evaluate...")
from pipeline.join_and_evaluate import run_join_and_evaluate
merged, metrics = run_join_and_evaluate()

total = time.time() - t0
print(f"\nPipeline complete in {total:.1f}s")
print(f"  Variants: {metrics['n_variants']}")
print(f"  With SGE: {metrics['n_with_sge']}")
print(f"  AUROC:    {metrics['auroc']}")

## 6. Sanity check

In [ ]:
df = pd.read_csv("data/variant_scores.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nESM score stats:")
print(df["esm_score"].describe())
print(f"\nSGE coverage: {df['sge_score'].notna().sum()} / {len(df)}")
print(f"\nSample rows:")
df.head(10)

## 7. Download output files

Copy these back to your local repo's `data/` folder.

In [ ]:
try:
    from google.colab import files
    files.download("data/variant_scores.csv")
    files.download("data/metrics.json")
    # Also download ESM cache so you don't need to rerun
    files.download("cache/esm_scores.csv")
    print("Downloads triggered!")
except ImportError:
    print("Not running in Colab. Files are at:")
    print("  data/variant_scores.csv")
    print("  data/metrics.json")
    print("  cache/esm_scores.csv")